# Streaming FAC — the lookahead sweep on GPU

Trains the **causal accent translator** at 7 lookaheads × 2 targets and
produces the RQ1 (knee) and RQ3 (conversion vs transcription) curves.

## What is being trained, and why this task

`KoelLabs/L2Arctic` gives per utterance: audio, `g2p` (the **canonical**
phone sequence a native speaker would produce) and `ipa` (the **produced**
sequence this L2 speaker actually said). It does **not** ship native
reference audio, so PHONOS's golden-target recipe isn't available from this
release alone.

But the supervised core of PHONOS is available directly:

> *causal accent translator: non-native audio → **native** phone sequence*

CTC against `g2p`. That is PHONOS's own description of its translator, and it
is precisely the component RQ1 asks about.

**The control that makes RQ3 answerable.** Identical architecture, capacity,
data, seed and step count — swap only the target tensor:

| arm | target | task |
|---|---|---|
| `native` | `g2p` | accent **conversion** — decide what the speaker *should* have said |
| `produced` | `ipa` | accent-faithful **transcription** — report the local gesture |

H3 predicts the conversion arm benefits more from lookahead. Two arms that
differ in one tensor is a cleaner control than AC-vs-VC-only.

**Cost:** a CTC head over a frozen encoder, not a generative pipeline. ~30–60
min per condition on a T4 ⇒ the full 14 runs is a single day, not 200 GPU-hours.

In [ ]:
#@title 1. Assert and record the GPU
import subprocess, json, os
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
import torch
assert torch.cuda.is_available(), 'Runtime -> Change runtime type -> GPU, then Restart.'
GPU = {'name': torch.cuda.get_device_name(0),
       'mem_gb': round(torch.cuda.get_device_properties(0).total_memory/1024**3, 1),
       'torch': torch.__version__, 'cuda': torch.version.cuda}
print(json.dumps(GPU, indent=2))
print('\nRecord this. A GPU latency row without a GPU name is not a result.')

In [ ]:
#@title 2. Clone the repo (+ optional Drive for checkpoints)
GIT_URL = 'https://github.com/Dipeshtripathi13/streaming-fac-lookahead.git'  #@param {type:'string'}
USE_DRIVE = True  #@param {type:'boolean'}
import os
REPO = '/content/research'
!rm -rf {REPO} && git clone -q {GIT_URL} {REPO} && echo cloned
if USE_DRIVE:
    from google.colab import drive; drive.mount('/content/drive')
    DRIVE = '/content/drive/MyDrive/accent_con'
    CKPT, RES = f'{DRIVE}/checkpoints', f'{DRIVE}/results'
else:
    CKPT, RES = '/content/checkpoints', '/content/results'
for d in (CKPT, RES): os.makedirs(d, exist_ok=True)
%cd {REPO}
!ls

In [ ]:
#@title 3. Install + Hugging Face auth
!pip -q install soundfile transformers datasets huggingface_hub 2>&1 | tail -2
# KoelLabs/L2Arctic is GATED (CC-BY-NC-4.0). Accept the terms once at
#   https://huggingface.co/datasets/KoelLabs/L2Arctic
# then paste a read token below. Non-commercial licence -- relevant to the
# model-weights release decision in the proposal's deliverables.
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
#@title 4. Sanity gates -- all must pass before spending GPU time
%cd {REPO}
!python tests/test_causal.py
!python tests/test_pipeline_invariants.py
!python -c "import sys;sys.path.insert(0,'src');import runpy;runpy.run_module('sfac.translator',run_name='__main__')"
!python bench/hardware_probe.py --out {RES}/hw_gpu.json | tail -20

## 5. The causality proof — run this before believing any lookahead label

**WavLM is not causal just because you masked its attention.** There are
**two** leaks underneath the transformer stack, and fixing either alone is
not enough:

1. **`pos_conv_embed`** — depthwise Conv1d, kernel 128, symmetric padding →
   at a 20 ms frame rate, **1.28 s of future** in every frame.
2. **The feature-encoder GroupNorm** — base checkpoints use
   `feat_extract_norm="group"`, normalising each channel over the **entire
   utterance**. Every output frame depends on every input frame: unbounded,
   not merely long. (`-large` uses `feat_extract_norm="layer"` and is safe.)

Measured on `wavlm-base-plus` (M4, 2 Aug 2026), relative L2 under a truncation
test — delete audio after frame *t*, check whether an earlier frame moved:

| configuration | relative L2 | causal? |
|---|---:|---|
| attention mask only | 1.14e-2 | no |
| + causal positional conv | 6.00e-3 | no |
| + cumulative GroupNorm | **2.60e-2** | no |
| **both patches** | **6.14e-6** | **yes** |

Note the third row: **patching GroupNorm alone makes it worse.** A partial
causality fix is not a partial improvement.

The next cell re-runs all four ablations on your GPU box. If `mask_both` does
not come back causal, **stop** — every lookahead label in the sweep would be
wrong, and the trainer will abort anyway.

In [ ]:
#@title 5. Causality self-test
!python bench/bench_content_degradation.py --selftest --tag gpu

## 6. THE JOB: dense-L pilot — is a narrow knee hiding between the samples?

On the M4, 48 utterances at L ∈ {0,20,40,80,160,320,640} gave a **log-linear**
curve: R² = 0.9955 against log₂(L), constant −0.081 per doubling, **no knee**.

The one caveat that could overturn that: 7 geometric points cannot see a knee
narrower than an octave. So sample **every 10 ms from 0 to 200 ms** (where any
knee would plausibly sit, given PHONOS's 40 ms and TVTSyn's 80 ms) on 200
utterances instead of 48.

Two outcomes, both worth having:
* still log-linear → the no-knee finding is solid at 10 ms resolution, and RQ1's
  answer becomes "publish the exchange rate, there is no operating point";
* a knee appears → we have its location at 10 ms resolution, which is a
  stronger result than the original 7-point grid could ever have given.

~5 min on a T4.

In [ ]:
#@title 6. Dense-L pilot
!python bench/bench_content_degradation.py --dense --n-utts 200 --device cuda --tag gpu_dense
!cp results/raw/content_degradation_gpu_dense* {RES}/ 2>/dev/null; ls {RES}

In [ ]:
#@title 6b. Read the answer
import json, sys
sys.path.insert(0,'eval')
d = json.load(open('results/raw/content_degradation_gpu_dense_summary.json'))
k = d['knee_divergence']
print(f"n_utts={d['n_utts']}  grid={len(d['lookaheads_ms'])} points "
      f"({min(d['lookaheads_ms'])}..{max(d['lookaheads_ms'])} ms)")
print(f"R2 log-linear = {k['r2_loglinear']}   R2 linear = {k['r2_linear']}")
print(f"has_knee = {k['has_knee']}")
print(k['interpretation'])
print(f"\n(naive chord estimator would have said {k['chord_argmax_ms']:.0f} ms)")
h = d.get('h2_proxy_paired')
if h: print(f"\nH2 proxy: {h['mean_diff']:+.4f}  CI95 {h['ci95']}  sig={h['significant']}")

In [ ]:
#@title 7. Standard-grid pilot (reproduces the M4 run on GPU) -- the RQ1 lower bound
# If the ENCODER has already lost the information at L=0, no downstream
# converter can recover it. This is a lower bound on the whole system's
# lookahead requirement, obtainable without training anything.
!python bench/bench_content_degradation.py --n-utts 120 --device cuda --tag gpu
!cp results/raw/content_degradation_gpu* {RES}/ 2>/dev/null; ls {RES}

In [ ]:
#@title 7. Smoke test the training pipeline (~3 min)
!python train/train_translator.py --smoke --device cuda --out {RES}/translator_smoke

In [ ]:
#@title 8. THE SWEEP -- 7 lookaheads x 2 targets
# T4: budget ~6-10 h. Colab free tier reclaims sessions, so either use Pro+
# background execution or run it in two halves (--targets native, then produced).
!python train/train_translator.py \
    --lookaheads 0 20 40 80 160 320 640 \
    --targets native produced \
    --steps 8000 --batch-size 8 --device cuda \
    --ckpt-dir {CKPT} \
    --out {RES}/translator_sweep
!cp {RES}/translator_sweep* {REPO}/results/raw/ 2>/dev/null; echo done

In [ ]:
#@title 9. Read the curves
import json
s = json.load(open(f'{RES}/translator_sweep_summary.json'))
for arm in ('native','produced'):
    c = s.get(f'curve_{arm}')
    if not c: continue
    print(f"\n{arm}:")
    for L, p in zip(c['lookaheads_ms'], c['test_per']):
        print(f"  L={L:>4.0f} ms   test PER {p:.4f}")
    print(f"  knee {c['knee']['knee_ms']:.0f} ms   "
          f"relative gain 0->640 {c['relative_gain_0_to_max']:.3f}")
if 'H3' in s:
    h = s['H3']
    print(f"\nH3 {'SUPPORTED' if h['supported'] else 'NOT supported'}: "
          f"conversion gain {h['gain_conversion']:.3f} vs "
          f"transcription gain {h['gain_transcription']:.3f}")
    print(h['reading'])

In [ ]:
#@title 10. GPU latency row -- CUDA-event timed
# CUDA kernels are asynchronous. Wall-clock around a forward pass measures
# launch overhead, not execution. This is the most common way published GPU
# latency numbers are wrong, and it is worth a sentence in the methods.
import torch, csv, sys
sys.path.insert(0, f'{REPO}/src')
from sfac.translator import TranslatorConfig, Target, PhoneVocab, build_translator
from sfac.latency import LatencyBudget

vocab = PhoneVocab([chr(97+i) for i in range(60)])
rows = []
for L in (0, 20, 40, 80, 160, 320, 640):
    cfg = TranslatorConfig(lookahead_ms=L, target=Target.NATIVE)
    m, info = build_translator(cfg, vocab); m = m.cuda().eval()
    g = cfg.geometry
    T = (g.lookback_frames or 100) + g.chunk_frames + g.lookahead_frames
    x = torch.randn(1, T, cfg.d_model, device='cuda')
    with torch.no_grad():
        for _ in range(10): m(x)
        torch.cuda.synchronize()
        ms = []
        for _ in range(50):
            s, e = torch.cuda.Event(True), torch.cuda.Event(True)
            s.record(); m(x); e.record(); torch.cuda.synchronize()
            ms.append(s.elapsed_time(e))
    ms.sort()
    b = LatencyBudget(chunk_ms=g.chunk_ms, lookahead_ms=L,
                      compute_ms_p50=ms[len(ms)//2], compute_ms_p95=ms[int(.95*len(ms))],
                      label=f'gpu/L{L}')
    r = b.to_row(); r.pop('meta.per_stage', None)
    r.update({'hw_class':'gpu','gpu':GPU['name'],'params':info['params'],
              'T_kv':T,'timing':'cuda_event','preset':'translator'})
    rows.append(r)
    print(f"L={L:>4} t_algo={b.algorithmic_ms:>6.0f}ms  "
          f"t_cmp_p50={b.compute_ms_p50:>7.3f}ms  RTF={b.rtf_p50:.4f}")

out = f'{RES}/gpu_latency_sweep.csv'
with open(out,'w',newline='') as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0])); w.writeheader(); w.writerows(rows)
print('\nwrote', out, '-- copy to research/results/raw/ and re-run bench/make_figures.py')

## Notes

**Checkpointing.** `--ckpt-dir` writes to Drive, one file per condition, each
carrying its config fingerprint and lookahead. A reclaimed session costs one
condition, not the sweep.

**Don't keep-alive with a JS click injector.** It violates Colab's ToS and
gets accounts limited. Pro+ background execution is the supported mechanism;
for anything longer, rent — see `setup/SETUP_GPU_COLAB.md` §B (RTX 4090 at
~$0.34/hr beats Colab Pro+ per useful hour above ~100 h).

**Frozen encoder is the default.** That makes the sweep a question about how
much context the *converter* needs given a fixed representation. `--unfreeze`
lets the encoder re-learn, which is a legitimate variant but must give every
condition identical steps or the comparison is confounded. Say which you ran.